In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration    
CONFIG = Configuration()

In [ ]:
import numpy as np

x_train = np.load("../data/ViolaJones/x_train.npy")
x_test = np.load("../data/ViolaJones/x_test.npy")


x = np.concatenate((x_train, x_test), axis=0)

# np.save("../data/ViolaJones/vpc_faces.npy", x)

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

no_faces_folder = "../data/ViolaJones/no_faces/vpc"
output_dir = Path(no_faces_folder)
output_dir.mkdir(parents=True, exist_ok=True)

for i, img in enumerate(x):
    arr = np.asarray(img)

    # Handle channel-first arrays: (C, H, W) -> (H, W, C)
    if arr.ndim == 3 and arr.shape[0] in (1, 3) and arr.shape[-1] not in (1, 3):
        arr = np.moveaxis(arr, 0, -1)

    # Convert to uint8 image range [0, 255]
    if arr.dtype != np.uint8:
        if np.issubdtype(arr.dtype, np.floating):
            if arr.size > 0 and arr.max() <= 1.0 and arr.min() >= 0.0:
                arr = (arr * 255.0).clip(0, 255).astype(np.uint8)
            else:
                arr = arr.clip(0, 255).astype(np.uint8)
        else:
            arr = arr.clip(0, 255).astype(np.uint8)

    # If image has shape (H, W, 1), save as grayscale 2D
    if arr.ndim == 3 and arr.shape[-1] == 1:
        arr = arr[:, :, 0]

    file_path = output_dir / f"no_face_{i:06d}.png"
    plt.imsave(file_path, arr, cmap="gray" if arr.ndim == 2 else None)

print(f"Saved {len(x)} images to {output_dir.resolve()}")

In [5]:
import cv2

def inspect_image_faces(image_path, face_cascade, scale_factor=1.1, min_neighbors=5, min_size=(24, 24)):
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        return None, None, False

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=min_size,
    )
    return image, faces, len(faces) > 0

def image_has_faces(image_path, face_cascade, scale_factor=1.1, min_neighbors=5, min_size=(24, 24)):
    _, _, has_faces = inspect_image_faces(
        image_path,
        face_cascade,
        scale_factor=scale_factor,
        min_neighbors=min_neighbors,
        min_size=min_size,
    )
    return has_faces

# Original dataset passing Open CV model

In [7]:
import os
cascade_path = os.path.join(CONFIG.cv_haar_cascades, "haarcascade_frontalface_default.xml")
face_cascade = cv2.CascadeClassifier(str(cascade_path))
if face_cascade.empty():
    raise FileNotFoundError(f"Could not load cascade XML: {cascade_path}")


In [ ]:
from tqdm import tqdm
import shutil
from maikol_utils.file_utils import list_dir_files, make_dirs


# Normalize configured output crop size into OpenCV (width, height) tuple
crop_size = (CONFIG.crop_size, CONFIG.crop_size)

# Output folders: images that pass vs fail the detector
passed_dir = os.path.join(CONFIG.faces_original_path, "cv_passed")
failed_dir = os.path.join(CONFIG.faces_original_path, "cv_failed")
make_dirs([passed_dir, failed_dir])

image_paths, _ = list_dir_files(CONFIG.faces_all_path)

passed = []
failed = []
unreadable = []

for img_path in tqdm(image_paths, desc="Processing images"):
    img, faces, has_faces = inspect_image_faces(img_path, face_cascade)
    if img is None:
        unreadable.append(img_path)
        continue

    if has_faces:
        # Keep only the largest detected face and overwrite output with resized crop
        x, y, w, h = max(faces, key=lambda r: r[2] * r[3])
        face_crop = img[y:y + h, x:x + w]

        if face_crop.size == 0:
            failed.append(img_path)
            shutil.copy2(img_path, os.path.join(failed_dir, os.path.basename(img_path)))
            continue

        interp = cv2.INTER_AREA if face_crop.shape[1] >= crop_size[0] and face_crop.shape[0] >= crop_size[1] else cv2.INTER_CUBIC
        face_crop_resized = cv2.resize(face_crop, crop_size, interpolation=interp)

        out_path = os.path.join(passed_dir, os.path.basename(img_path))
        ok = cv2.imwrite(out_path, face_crop_resized)
        if not ok:
            raise IOError(f"Failed to write cropped face to: {out_path}")

        passed.append(img_path)
    else:
        failed.append(img_path)
        shutil.copy2(img_path, os.path.join(failed_dir, os.path.basename(img_path)))

print(f"Source dir: {CONFIG.faces_all_path}")
print(f"Total candidate images: {len(image_paths)}")
print(f"Passed (>=1 face): {len(passed)}")
print(f"Failed (0 faces): {len(failed)}")
print(f"Unreadable: {len(unreadable)}")
print(f"Passed output (largest-face crops resized to {crop_size}): {passed_dir}")
print(f"Failed output: {failed_dir}")

# Keep lists available for downstream analysis in the notebook
passed_paths = [str(p) for p in passed]
failed_paths = [str(p) for p in failed]
unreadable_paths = [str(p) for p in unreadable]

Processing images: 100%|██████████| 136965/136965 [02:35<00:00, 879.81it/s]

Source dir: ../data/ViolaJones/face_images
Total candidate images: 136965
Passed (>=1 face): 67690
Failed (0 faces): 69275
Unreadable: 0
Passed output (largest-face crops resized to (24, 24)): ../data/ViolaJones/face_images/cv_passed
Failed output: ../data/ViolaJones/face_images/cv_failed


# Create partition

In [9]:
import numpy as np
CONFIG = Configuration(
    max_bg_samples=500,
    stride=1,
    max_faces=70000
    
)
rng = np.random.default_rng(seed=42)


In [7]:
from maikol_utils.file_utils import list_dir_files


all_faces, n = list_dir_files(CONFIG.faces_passed_path)
print(f"Total images in faces directory: {n}")

all_faces = rng.permutation(all_faces)

test_faces = all_faces[:int(CONFIG.test_size * len(all_faces))]
train_faces = all_faces[int(CONFIG.test_size * len(all_faces)):]
test_faces = rng.choice(test_faces, size=min(len(test_faces), CONFIG.max_faces), replace=False)
train_faces = rng.choice(train_faces, size=min(len(train_faces), CONFIG.max_faces), replace=False)

print(f"Train faces: {len(train_faces)}")
print(f"Test faces: {len(test_faces)}")

Total images in faces directory: 67690
Train faces: 60921
Test faces: 6769


In [8]:
from tqdm import tqdm 
import shutil
import os

for split, paths in [("train", train_faces), ("test", test_faces)]:
    output_dir = CONFIG.faces_train_path if split == "train" else CONFIG.faces_test_path
    for img_path in tqdm(paths, desc=f"Copying {split} images"):
        shutil.copy2(img_path, os.path.join(output_dir, os.path.basename(img_path)))
    print(f"Copied {len(paths)} images to {output_dir}")

Copying train images: 100%|██████████| 60921/60921 [00:01<00:00, 40732.33it/s]


Copied 60921 images to ../data/ViolaJones/face_images/train


Copying test images: 100%|██████████| 6769/6769 [00:00<00:00, 9874.99it/s] 

Copied 6769 images to ../data/ViolaJones/face_images/test


# No faces dataset

In [11]:
from maikol_utils.file_utils import list_dir_files

all_face_images, n = list_dir_files(CONFIG.faces_all_path, max_files=7000)
print(f"Total images in faces_all_path: {n}")
print(f"Images without faces will be copied to: {CONFIG.no_faces_all_path}")

Total images in faces_all_path: 7000
Images without faces will be copied to: ../data/ViolaJones/no_faces/all


In [12]:
from tqdm import tqdm
import os
import shutil
from maikol_utils.file_utils import make_dirs

make_dirs([CONFIG.no_faces_all_path])

no_face_images = []
with_face_images = []
unreadable_images = []

for img_path in tqdm(all_face_images, desc="Filtering images without faces"):
    img, faces, has_faces = inspect_image_faces(img_path, face_cascade)
    if img is None:
        unreadable_images.append(img_path)
        continue

    if has_faces:
        with_face_images.append(img_path)
        continue

    source_path = str(img_path)
    target_path = os.path.join(CONFIG.no_faces_all_path, os.path.basename(source_path))
    shutil.copy2(source_path, target_path)
    no_face_images.append(img_path)

print(f"Images with faces skipped: {len(with_face_images)}")
print(f"Images without faces copied: {len(no_face_images)}")
print(f"Unreadable images skipped: {len(unreadable_images)}")
print(f"Copied no-face images to: {CONFIG.no_faces_all_path}")

Filtering images without faces: 100%|██████████| 7000/7000 [00:07<00:00, 940.76it/s] 

Images with faces skipped: 3681
Images without faces copied: 3319
Unreadable images skipped: 0
Copied no-face images to: ../data/ViolaJones/no_faces/all


In [11]:
from src.data import load_gb_images
from maikol_utils.file_utils import list_dir_files

# test_bg = load_gb_images(CONFIG, partition="test")
test_bg = list_dir_files('../data/ViolaJones/no_faces/open-images-v7/test/data')[0]

In [12]:
from tqdm import tqdm 
from src.data import get_all_image_crops

all_crops = []

for img_path in tqdm(test_bg, desc="Extracting crops from background images"):
    crops = get_all_image_crops(CONFIG, img_path)
    print(f"Extracted {len(crops)} crops from {img_path}")
    all_crops.extend(crops)

len(all_crops)

Extracting crops from background images:   0%|          | 0/500 [00:00<?, ?it/s]

Extracting crops from background images:   0%|          | 1/500 [00:01<09:47,  1.18s/it]

Extracted 1615081 crops from ../data/ViolaJones/no_faces/open-images-v7/test/data/00a5f08a8fdc1cdb.jpg


Extracting crops from background images:   0%|          | 1/500 [00:02<17:36,  2.12s/it]


KeyboardInterrupt: 

In [ ]:
all_crops = rng.choice(all_crops, 100_000, replace=False)
all_crops = [crop['img'] for crop in all_crops]

In [ ]:
all_crops[5].shape

In [ ]:
import os
import matplotlib.pyplot as plt

for idx, crop in tqdm(enumerate(all_crops), total=len(all_crops), desc="Saving crops to disk"):
    # safe on disk
    path = os.path.join(CONFIG.no_faces_crops_path, f"crop_{idx:06d}.png")
    plt.imsave(path, crop)

---